# Лекция 1. Асинхронность и конкурентность в Python

## О чём лекция

Представьте: программа делает 10 HTTP-запросов по очереди.
Каждый занимает 1 секунду. Итого 10 секунд. Но процессор простаивает ~9.9 секунд
из 10 — он просто ждёт ответ от сети.

**Проблема не в скорости CPU, а в том, что программа стоит и ждёт.**

В этой лекции разберём:
1. Почему CPU простаивает (I/O-bound vs CPU-bound)
2. Три подхода: `threading`, `multiprocessing`, `asyncio`
3. Когда что применять

---

## 1. Два типа задач

Одни задачи упираются в скорость CPU, другие — в скорость внешних устройств.
От этого зависит, какой инструмент нам поможет.

| Тип | Что происходит | Примеры | CPU |
|-----|---------------|---------|-----|
| **I/O-bound** | Программа **ждёт** внешнее устройство | HTTP-запросы, чтение файлов, SQL-запросы, SNMP-опрос | Свободен |
| **CPU-bound** | Программа **считает** | Обработка изображений, сжатие, парсинг, расчёт BGP-метрик | Занят |

**Аналогия:** повар на кухне. Пока суп варится (I/O), можно резать салат.
Но если нужно перебрать 100 кг гречки (CPU), переключение не поможет —
нужна вторая пара рук.

Проверим на практике — запустим пример:

In [ ]:
# Запуск из корня проекта:
#   python lectures/01_lecture/examples/01_sync/02_io_vs_cpu.py

# Пример показывает два типа задач side-by-side:
# - I/O-bound: 3 HTTP-запроса (ждём ответ, CPU свободен)
# - CPU-bound: вычисление простых чисел (CPU загружен)

print(
    "Смотрите в терминале: python lectures/01_lecture/examples/01_sync/02_io_vs_cpu.py"
)

---

## 2. Что такое конкурентность

**Конкурентность (concurrency)** — выполнение нескольких задач с перекрытием
во времени. Задачи не обязательно выполняются одновременно — но их выполнение
накладывается.

| | Конкурентность | Параллелизм |
|---|---|---|
| Суть | Задачи **выглядят** одновременными | Задачи **реально** выполняются одновременно |
| Сколько ядер нужно | Достаточно 1 ядра | Нужно ≥ 2 ядер |
| Пример в Python | `threading`, `asyncio` | `multiprocessing` |

**Конкурентность — про структуру программы** (как переключаемся между задачами).
**Параллелизм — про исполнение** (сколько задач физически выполняется в момент).

---

## 3. Многопоточность — threading

### Как работает

**Поток (thread)** — линейный путь выполнения кода внутри процесса.
Несколько потоков делят память процесса. ОС может переключить любой поток
в любой момент (вытесняющая многозадачность).

```python
import threading, time

def work(name: str, delay: int) -> None:
    time.sleep(delay)
    print(f"[{name}] done in {delay}s")

t1 = threading.Thread(target=work, args=("A", 3))
t2 = threading.Thread(target=work, args=("B", 2))
t1.start(); t2.start()
t1.join(); t2.join()  # ждём завершения
```

### GIL (Global Interpreter Lock)

GIL — блокировка CPython, разрешающая выполнять **только один поток
байт-кода** одновременно. Это важно:

- **При I/O** (сеть, диск) — GIL **отпускается**, другие потоки работают
- **При CPU-bound** — GIL **не отпускается**, поток блокирует всех

### ThreadPoolExecutor — удобная обёртка

```python
from concurrent.futures import ThreadPoolExecutor, as_completed

with ThreadPoolExecutor(max_workers=10) as pool:
    # dict-to-future: каждый Future → URL/устройство
    future_to_url = {pool.submit(fetch, url): url for url in urls}
    for future in as_completed(future_to_url):  # по мере готовности
        url = future_to_url[future]
        print(f"{url}: {future.result()}")
```

`as_completed()` отдаёт результаты **по мере готовности**, не в порядке запуска.
Быстрые запросы приходят первыми.

In [ ]:
# Примеры для запуска:
#   python lectures/01_lecture/examples/02_threading/01_simple_thread.py
#      -> базовые потоки, join(), сравнение sequential vs threading
#
#   python lectures/01_lecture/examples/02_threading/02_thread_pool.py
#      -> ThreadPoolExecutor + as_completed
#
#   python lectures/01_lecture/examples/02_threading/03_race_condition.py
#      -> race condition: counter += 1 не атомарно, нужен Lock

print("Примеры запускаются в терминале. Рекомендуемый порядок:")
print("  1. python lectures/01_lecture/examples/02_threading/01_simple_thread.py")
print("  2. python lectures/01_lecture/examples/02_threading/02_thread_pool.py")
print("  3. python lectures/01_lecture/examples/02_threading/03_race_condition.py")

---

## 4. Многопроцессорность — multiprocessing

### Зачем

Threading **не ускоряет** CPU-bound задачи из-за GIL. Для этого — процессы:

- У каждого процесса свой интерпретатор Python, своя память, свой GIL
- Процессы выполняются **параллельно** на разных ядрах CPU

```python
from multiprocessing import Process

def worker(n: int) -> int:
    return sum(i * i for i in range(n))

if __name__ == "__main__":
    procs = [Process(target=worker, args=(10_000_000,)) for _ in range(4)]
    for p in procs: p.start()
    for p in procs: p.join()
```

### Простое правило

| Задача | threading | multiprocessing |
|--------|:---------:|:---------------:|
| I/O-bound (HTTP, SQL, SNMP) | ✅ Ускоряет | ✅ Тоже, но тяжеловесно |
| CPU-bound (расчёты, парсинг) | ❌ Бесполезно (GIL) | ✅ Параллелизм на ядрах |

In [ ]:
# Наглядная демонстрация: threading НЕ ускоряет CPU-bound, а multiprocessing — ускоряет
#   python lectures/01_lecture/examples/03_multiprocessing/02_cpu_bound.py

print(
    "Запустите: python lectures/01_lecture/examples/03_multiprocessing/02_cpu_bound.py"
)
print(
    "Ожидаемый результат: threading ~ синхронного, multiprocessing ~ N ядер × ускорение"
)

---

## 5. Асинхронность — asyncio

### Как работает

**Кооперативная многозадачность:** один поток, одно ядро, программа **сама решает**,
когда переключиться (через `await`). Никто её не прерывает.

```python
import asyncio

async def fetch(url: str) -> str:
    await asyncio.sleep(1.0)  # "я жду, займись другими"
    return f"data:{url}"

async def main():
    results = await asyncio.gather(
        fetch("site1"), fetch("site2"), fetch("site3"),
    )
    print(results)  # ~1 с, не 3 с

asyncio.run(main())
```

### Event loop — «диспетчер» корутин

**Event loop (цикл событий)** — главный компонент asyncio. Это while True,
который крутится в одном потоке и делает три вещи:

1. **Хранит очередь корутин**, готовых к выполнению
2. **Берёт одну корутину**, запускает её до первого `await`
3. **Переключается на другую**, когда текущая сказала `await`

Как только все корутины завершены — event loop останавливается.

```python
# Упрощённая логика event loop:
ready_queue = [coro1, coro2, coro3]

while ready_queue:
    coro = ready_queue.pop(0)       # взять корутину
    coro.send(None)                 # выполнить до следующего await
    if not coro.done():
        ready_queue.append(coro)    # обратно в очередь
```

В реальности event loop сложнее: он умеет ждать файловые дескрипторы,
таймеры, сигналы. Но суть та же — **один поток, который эффективно
переключается между ожидающими задачами**.

`asyncio.run(main())` создаёт новый event loop, запускает `main()`,
и когда всё готово — закрывает loop.

### Future — «обещание» результата

**Future (фьючер, будущее)** — объект, который хранит **ещё не готовый результат**.
Корутина может подписаться: «когда результат появится — разбуди меня».

```python
from asyncio import Future

fut: Future[str] = Future()
# fut ещё не готов — в нём нет результата
fut.set_result("готово!")       # положили результат
result = await fut              # получили — мгновенно, результат уже есть
print(result)                   # готово!
```

Future — это **контейнер**, который в любой момент либо **завершён** (содержит
результат или исключение), либо **не завершён** (пустой).

#### Отношение к корутинам и Task

```python
import asyncio

async def work():
    return 42

task = asyncio.create_task(work())  # Task — наследник Future
print(isinstance(task, asyncio.Future))  # True
```

```
create_task(coro) → Task (подкласс Future)
                        ↓
                  await task  — получаем результат, когда корутина завершена
                  task.result()  — если уже завершена
                  task.done()    — True, если завершена
```

**Task** — это подкласс Future. Когда корутина заворачивается в `create_task()`,
она начинает выполняться в фоне, а Task — это Future, который станет завершённым,
когда корутина вернёт результат.

#### Future в ThreadPoolExecutor

Future есть не только в asyncio. `concurrent.futures.Future` работает так же,
но для потоков и процессов:

```python
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor() as pool:
    future: Future = pool.submit(fn, arg)  # Future — ещё не готов
    result = future.result()               # ждём и получаем
```

| | `concurrent.futures.Future` | `asyncio.Future` |
|---|---|---|
| Где живёт | В потоках/процессах | В event loop |
| Как ждут | `future.result()` (блокирует поток) | `await future` (не блокирует) |
| Наследники | — | `asyncio.Task` (корутина в фоне) |

**Аналогия:** Future — это квитанция в химчистке. Вы сдали вещь (запустили задачу)
и получили номерок. По номерку можно получить вещь, когда она готова.
Пока не готова — занимаетесь другими делами.

### Ключевые понятия

| Конструкция | Что делает |
|---|---|
| `async def fn()` | Определяет **корутину** (функцию, которую можно приостановить) |
| `await` | Говорит event loop: "я жду, займись другими корутинами" |
| `asyncio.gather(c1, c2, c3)` | Запускает несколько корутин **одновременно** и ждёт все |
| `asyncio.create_task(coro)` | Запускает корутину **в фоне**, не дожидаясь завершения |
| `asyncio.Future()` | Контейнер для **ещё не готового результата** |
| `asyncio.run(main())` | Точка входа: создаёт event loop и запускает главную корутину |

### Блокирующий код внутри asyncio

`time.sleep()` **блокирует весь event loop**. Используйте `await asyncio.sleep()`.
Для блокирующего кода — `asyncio.to_thread(fn, arg)` (выполнит в отдельном потоке).

In [ ]:
# Примеры по нарастающей:
#
# 1. Базовая корутина:
#    python lectures/01_lecture/examples/04_asyncio/01_basic_coroutine.py
#    -> async def / await, sequential vs gather
#
# 2. asyncio.gather:
#    python lectures/01_lecture/examples/04_asyncio/02_gather.py
#    -> параллельный запуск нескольких корутин, return_exceptions
#
# 3. Задачи (Tasks):
#    python lectures/01_lecture/examples/04_asyncio/03_tasks.py
#    -> create_task — фоновая работа

print("Рекомендуемый порядок:")
print("  1. python lectures/01_lecture/examples/04_asyncio/01_basic_coroutine.py")
print("  2. python lectures/01_lecture/examples/04_asyncio/02_gather.py")
print("  3. python lectures/01_lecture/examples/04_asyncio/03_tasks.py")

---

## 6. Сочетание threading и asyncio

Asyncio отлично управляет тысячами корутин, но он живёт **в одном потоке**.
Что делать, если внутри asyncio-программы нужно выполнить блокирующий код?

```python
time.sleep(5)  # блокирует ВЕСЬ event loop — все корутины встанут!
```

**Решение: выгрузить блокирующий код в пул потоков.**

### Как это работает

Event loop продолжает крутиться в основном потоке. Блокирующая функция
выполняется в ThreadPoolExecutor. Пока поток спит (time.sleep), GIL отпущен,
и event loop может переключаться между корутинами.

```
Время:  0s────────1s────────2s────────3s
─────
Поток 1 (event loop):  ▓▓ корутина A ▓▓░░░ корутина B ░░░▓▓ корутина C ▓▓
Поток 2 (to_thread):   ░░░░░░░░ time.sleep(1) ░░░░░░░░░░░░░░░░░░░░░░░░░░░░
Поток 3 (to_thread):   ░░░░░░░░░░░░░░░░ time.sleep(0.8) ░░░░░░░░░░░░░░░░░░
```

### Три способа выгрузки

| Способ | Когда использовать |
|---|---|
| `asyncio.to_thread(fn, arg)` | Простой случай, пул по умолчанию |
| `loop.run_in_executor(pool, fn, arg)` | Нужен свой пул с max_workers |
| `run_in_executor(ProcessPoolExecutor)` | CPU-bound код (обход GIL) |

### asyncio.to_thread — быстрый старт

```python
import asyncio

def blocking_read(filename: str) -> str:
    with open(filename) as f:
        return f.read()

async def main():
    data = await asyncio.to_thread(blocking_read, "big_file.txt")
    print(f"Прочитано {len(data)} байт без блокировки event loop")
```

`to_thread` использует **пул по умолчанию** (min(32, cpu_count + 4) потоков).
Этого достаточно для большинства задач.

### loop.run_in_executor — полный контроль

```python
from concurrent.futures import ThreadPoolExecutor

async def main():
    loop = asyncio.get_running_loop()
    with ThreadPoolExecutor(max_workers=2) as pool:
        tasks = [
            loop.run_in_executor(pool, blocking_io, name, delay)
            for name, delay in [("A", 1), ("B", 0.5), ("C", 0.3)]
        ]
        results = await asyncio.gather(*tasks)
```

Полезно когда:
- Нужно **ограничить** число потоков (не перегрузить базу)
- Использовать **один пул** для нескольких вызовов (переиспользование)
- Нужен **ProcessPoolExecutor** для CPU-bound

### ProcessPoolExecutor — параллелизм CPU

```python
from concurrent.futures import ProcessPoolExecutor

async def main():
    loop = asyncio.get_running_loop()
    with ProcessPoolExecutor(max_workers=4) as pool:
        results = await asyncio.gather(*[
            loop.run_in_executor(pool, heavy_cpu, n) for n in big_numbers
        ])
```

Каждый процесс имеет **свой GIL** — вычисления идут параллельно на ядрах.
Разница с ThreadPoolExecutor для CPU-bound может быть 2–4×.